In [1]:
import pandas as pd
import os
import tensorflow as tf
import numpy as np
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.models import load_model
from sklearn.metrics import roc_auc_score, accuracy_score

In [2]:
from google.colab import drive
drive.mount('/content/drive')

!rm -rf /content/merged_dataset
!rm -rf /content/test_dataset
!unzip -q "/content/drive/MyDrive/MRP/merged_dataset.zip" -d "/content/merged_dataset"
!unzip -q "/content/drive/MyDrive/MRP/test_dataset.zip" -d "/content/test_dataset"

Mounted at /content/drive


In [3]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    "/content/merged_dataset/merged_dataset",
    labels="inferred",
    image_size=(256, 256),
    batch_size=64,
    validation_split=0.2,
    subset="training",
    seed=42
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    "/content/merged_dataset/merged_dataset",
    labels="inferred",
    image_size=(256, 256),
    batch_size=64,
    validation_split=0.2,
    subset="validation",
    seed=42
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    "/content/test_dataset/test_dataset",
    labels="inferred",
    image_size=(256, 256),
    batch_size=64
)

Found 351500 files belonging to 2 classes.
Using 281200 files for training.
Found 351500 files belonging to 2 classes.
Using 70300 files for validation.
Found 18124 files belonging to 2 classes.


In [4]:
def fft_magnitude(image):
    # Convert to float32
    image = tf.image.convert_image_dtype(image, tf.float32)

    # Compute FFT per channel
    fft = tf.signal.fft2d(tf.cast(image, tf.complex64))

    # Shift zero-frequency to center
    fft_shift = tf.signal.fftshift(fft)

    # Magnitude
    magnitude = tf.abs(fft_shift)

    # Log scale (critical for stability)
    magnitude = tf.math.log(magnitude + 1e-8)

    # Normalize to [0,1]
    magnitude = (magnitude - tf.reduce_min(magnitude)) / (tf.reduce_max(magnitude) - tf.reduce_min(magnitude) + 1e-8)

    return magnitude

In [5]:
train_fft = train_ds.map(lambda x, y: (fft_magnitude(x), y), num_parallel_calls=tf.data.AUTOTUNE)
val_fft = val_ds.map(lambda x, y: (fft_magnitude(x), y), num_parallel_calls=tf.data.AUTOTUNE)
test_fft = test_ds.map(lambda x, y: (fft_magnitude(x), y), num_parallel_calls=tf.data.AUTOTUNE)

train_fft = train_fft.prefetch(tf.data.AUTOTUNE)
val_fft = val_fft.prefetch(tf.data.AUTOTUNE)
test_fft = test_fft.prefetch(tf.data.AUTOTUNE)

In [6]:
def dct_full_image(image):
    # Convert to float32
    image = tf.image.convert_image_dtype(image, tf.float32)

    # Apply 2D DCT per channel
    # DCT over width
    x = tf.signal.dct(image, type=2, norm='ortho')

    # DCT over height
    x = tf.signal.dct(tf.transpose(x, perm=[1, 0, 2]), type=2, norm='ortho')
    x = tf.transpose(x, perm=[1, 0, 2])

    # Magnitude
    x = tf.abs(x)

    # Normalize
    x_min = tf.reduce_min(x)
    x_max = tf.reduce_max(x)
    x = (x - x_min) / (x_max - x_min + 1e-8)

    return x


def dct_full_batch(batch):
    return tf.map_fn(
        dct_full_image,
        batch,
        fn_output_signature=tf.float32
    )

In [7]:
train_dct = train_ds.map(lambda x, y: (dct_full_batch(x), y), num_parallel_calls=tf.data.AUTOTUNE)
val_dct   = val_ds.map(lambda x, y: (dct_full_batch(x), y), num_parallel_calls=tf.data.AUTOTUNE)
test_dct  = test_ds.map(lambda x, y: (dct_full_batch(x), y), num_parallel_calls=tf.data.AUTOTUNE)

train_dct = train_dct.prefetch(tf.data.AUTOTUNE)
val_dct   = val_dct.prefetch(tf.data.AUTOTUNE)
test_dct  = test_dct.prefetch(tf.data.AUTOTUNE)

In [9]:
# Load base models and build feature extractors
rgb_model = load_model("/content/drive/MyDrive/MRP/models/raw_cnn_baseline.h5")
rgb_extractor = tf.keras.Model(
    inputs=rgb_model.layers[0].input,
    outputs=rgb_model.get_layer("feature_vector").output
)

fft_model = load_model("/content/drive/MyDrive/MRP/models/fft_cnn_baseline.h5")
fft_extractor = tf.keras.Model(
    inputs=fft_model.layers[0].input,
    outputs=fft_model.get_layer("fft_feature_vector").output
)

dct_model = load_model("/content/drive/MyDrive/MRP/models/dct_cnn_baseline.h5")
dct_extractor = tf.keras.Model(
    inputs=dct_model.layers[0].input,
    outputs=dct_model.get_layer("dct_feature_vector").output
)

In [10]:
# Helper: extract embeddings from a tf.data.Dataset
def extract_embeddings(extractor, dataset):
    all_emb = []
    all_y = []
    for batch_x, batch_y in dataset:
        emb = extractor(batch_x, training=False)  # (batch, feat_dim)
        all_emb.append(emb.numpy())
        all_y.append(batch_y.numpy())
    return np.vstack(all_emb), np.hstack(all_y)

In [11]:
# Extract embeddings for train / val / test
# RGB uses raw datasets
X_rgb_train, y_train = extract_embeddings(rgb_extractor, train_ds)
X_rgb_val,   y_val   = extract_embeddings(rgb_extractor, val_ds)
X_rgb_test,  y_test  = extract_embeddings(rgb_extractor, test_ds)

# FFT uses fft datasets
X_fft_train, _ = extract_embeddings(fft_extractor, train_fft)
X_fft_val,   _ = extract_embeddings(fft_extractor, val_fft)
X_fft_test,  _ = extract_embeddings(fft_extractor, test_fft)

# DCT uses dct datasets
X_dct_train, _ = extract_embeddings(dct_extractor, train_dct)
X_dct_val,   _ = extract_embeddings(dct_extractor, val_dct)
X_dct_test,  _ = extract_embeddings(dct_extractor, test_dct)

# Concatenate embeddings
X_train = np.concatenate([X_rgb_train, X_fft_train, X_dct_train], axis=1)
X_val   = np.concatenate([X_rgb_val,   X_fft_val,   X_dct_val],   axis=1)
X_test  = np.concatenate([X_rgb_test,  X_fft_test,  X_dct_test],  axis=1)

In [12]:
# Build dense meta-learner
input_dim = X_train.shape[1]  # should be 384

meta_model = models.Sequential([
    layers.Input(shape=(input_dim,)),

    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),

    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),

    layers.Dense(64, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.2),

    layers.Dense(1, activation='sigmoid')
])

meta_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [13]:
# Train meta-learner on embeddings
history = meta_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=64,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            patience=5,
            restore_best_weights=True
        )
    ]
)

Epoch 1/30
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 37s 6ms/step - accuracy: 0.8910 - loss: 0.2714 - val_accuracy: 0.9080 - val_loss: 0.2303
Epoch 2/30
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 11s 3ms/step - accuracy: 0.9059 - loss: 0.2348 - val_accuracy: 0.9101 - val_loss: 0.2245
Epoch 3/30
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 11s 3ms/step - accuracy: 0.9089 - loss: 0.2275 - val_accuracy: 0.9107 - val_loss: 0.2200
Epoch 4/30
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 11s 2ms/step - accuracy: 0.9102 - loss: 0.2236 - val_accuracy: 0.9116 - val_loss: 0.2175
Epoch 5/30
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 11s 2ms/step - accuracy: 0.9107 - loss: 0.2214 - val_accuracy: 0.9121 - val_loss: 0.2167
Epoch 6/30
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 11s 2ms/step - accuracy: 0.9120 - loss: 0.2196 - val_accuracy: 0.9123 - val_loss: 0.2159
Epoch 7/30
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 11s 2ms/step - accuracy: 0.9125 - loss: 0.2179 - val_accuracy: 0.9128 - val_loss: 0.2156
Epoch 8/30
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 11s 2ms/step - accuracy: 0.9129 - loss: 0

In [14]:
# Evaluate on held-out test set
test_loss, test_acc = meta_model.evaluate(X_test, y_test, verbose=0)
print("Meta-learner test accuracy:", test_acc)

y_prob = meta_model.predict(X_test, verbose=0)
print("Meta-learner test AUC:", roc_auc_score(y_test, y_prob))

Meta-learner test accuracy: 0.6604502201080322
Meta-learner test AUC: 0.767611880997404


In [23]:
def evaluate_model(model, dataset):
    y_true = []
    y_prob = []

    # Loop ensures predictions and labels stay aligned
    for x_batch, y_batch in dataset:
        preds = model.predict(x_batch, verbose=0).ravel()
        y_prob.append(preds)
        y_true.append(y_batch.numpy())

    y_true = np.concatenate(y_true)
    y_prob = np.concatenate(y_prob)

    # Convert probabilities to binary predictions
    y_pred = (y_prob >= 0.5).astype(int)

    # Compute metrics
    acc = accuracy_score(y_true, y_pred)
    auc = roc_auc_score(y_true, y_prob)

    return acc, auc

acc_rgb, auc_rgb = evaluate_model(rgb_model, test_ds)
print("RGB  - Accuracy:", acc_rgb, "AUC:", auc_rgb)

acc_fft, auc_fft = evaluate_model(fft_model, test_fft)
print("FFT  - Accuracy:", acc_fft, "AUC:", auc_fft)

acc_dct, auc_dct = evaluate_model(dct_model, test_dct)
print("DCT  - Accuracy:", acc_dct, "AUC:", auc_dct)

RGB  - Accuracy: 0.6422975060693004 AUC: 0.6451063764237438
FFT  - Accuracy: 0.6750165526373869 AUC: 0.4919693869932469
DCT  - Accuracy: 0.660560582652836 AUC: 0.6609828699721586


In [16]:
meta_model.save("/content/drive/MyDrive/MRP/models/meta_learning_ensemble_baseline.h5")